# AnomalyDINO-DPMM: Anomaly Detection with DINOv2 + Dirichlet Process Mixture Models

This notebook walks through the full pipeline step by step:

1. **Setup** — install dependencies and verify GPU
2. **Dataset** — download MVTec Grid (proxy for repeating industrial patterns)
3. **Data loading** — `IndustrialAnomalyDataset` supporting PNG and DICOM
4. **Feature extraction** — DINOv2 ViT-S/14 patch embeddings (32×32 = 1024 patches, dim 384)
5. **DP-GMM clustering** — variational Bayesian GMM, no manual K selection
6. **Anomaly scoring** — cosine distance to nearest centroid → heatmap

Based on: [Anomaly Detection by Clustering DINO Embeddings using a Dirichlet Process Mixture](https://arxiv.org/abs/2509.19997), MICCAI 2025.

In [2]:
# pydicom is needed for DICOM (.dcm) support.
# All other deps (torch, torchvision, sklearn, cv2, matplotlib, numpy) are pre-installed in Colab.
!pip install -q pydicom 

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Hello")
if device.type == "cpu":
    print("WARNING: No GPU detected. DINOv2 inference will be slow.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

ModuleNotFoundError: No module named 'torch'

## Step 1: Dataset Setup

MVTec Grid contains 264 defect-free training images of a woven textile with a repeating grid pattern.
The test set has 5 defect categories (bent, broken, glue, metal_contamination, thread) plus good samples.

**Upload options (pick one):**
- **Google Drive (recommended):** Upload your local `grid/` folder to Google Drive once — it persists across sessions. Set `DRIVE_GRID_PATH` below to your Drive path.
- **Direct upload:** Zip your local `grid/` folder, upload via the cell below — lost on runtime reset.
- **Already present:** If you already uploaded the data, just run the cell — it skips uploading.

In [ ]:
import os
from google.colab import drive

EXTRACT_DIR = "dataset"
GRID_DIR = os.path.join(EXTRACT_DIR, "grid")

if os.path.exists(GRID_DIR):
    print(f"Dataset already present at: {GRID_DIR}")
else:
    # Mount Google Drive — opens a browser auth popup on first run
    drive.mount("/content/drive")

    # Set this to wherever you uploaded grid/ in your Drive
    # e.g. if you put it at My Drive/mvtec/grid/, set:
    #   DRIVE_GRID_PATH = "/content/drive/MyDrive/mvtec/grid"
    DRIVE_GRID_PATH = "/content/drive/MyDrive/coding/ML/datasets/mvtec/grid"  # <-- adjust this

    if not os.path.exists(DRIVE_GRID_PATH):
        raise FileNotFoundError(
            f"Could not find dataset at: {DRIVE_GRID_PATH}\n"
            "Upload your local grid/ folder to Google Drive, then update DRIVE_GRID_PATH above."
        )

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    os.symlink(DRIVE_GRID_PATH, GRID_DIR)
    print(f"Linked Drive dataset → {GRID_DIR}")

# Sanity check
train_good = os.path.join(GRID_DIR, "train", "good")
n_train = len([f for f in os.listdir(train_good) if f.endswith(".png")])
print(f"Training images (normal): {n_train}  (expected: 264)")
test_categories = sorted(os.listdir(os.path.join(GRID_DIR, "test")))
print(f"Test categories: {test_categories}")

## Step 2: Dataset Class

`IndustrialAnomalyDataset` supports both PNG and DICOM (16-bit) files.

Key design decisions:
- **Letterboxing** (not center-crop): preserves the full grid period geometry. Cropping would cut repeating units and bias the patch embeddings.
- **Percentile scaling** for DICOM: clips to the 1st–99th percentile to handle sparse X-ray dynamic range without clipping real structure.
- The label is derived from the top-level folder name (`good`, `bent`, etc.).

In [ ]:
import numpy as np
import cv2
import torch
import torchvision.transforms as T
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

try:
    from pydicom import dcmread
    DICOM_AVAILABLE = True
except ImportError:
    DICOM_AVAILABLE = False
    print("pydicom not available — DICOM files will not load. Run: !pip install pydicom")


class IndustrialAnomalyDataset(Dataset):
    def __init__(self, base_path, target_size=448):
        self.base_path = Path(base_path)
        self.target_size = target_size
        self.filepaths = []
        self.labels = []

        for f in self.base_path.rglob("*"):
            if f.is_file() and f.suffix.upper() in [".DCM", ".PNG", ".JPG", ".JPEG"]:
                self.filepaths.append(f)
                self.labels.append(f.relative_to(self.base_path).parts[0])

        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.filepaths)

    def preprocess_image(self, img_raw, is_dcm=False):
        img_float = img_raw.astype(np.float32)
        if is_dcm:
            p_low, p_high = np.percentile(img_float, (1.0, 99.0))
            if p_high <= p_low:
                p_high = p_low + 1e-6
            image = np.clip(img_float, p_low, p_high)
            image = (image - p_low) / (p_high - p_low)
        else:
            image = img_float / 255.0

        # Letterbox: aspect-preserving resize into a zero-padded square canvas
        ch, cw = image.shape[:2]
        scale = self.target_size / max(ch, cw)
        new_w, new_h = int(cw * scale), int(ch * scale)
        resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

        canvas = np.zeros((self.target_size, self.target_size), dtype=np.float32)
        x_off = (self.target_size - new_w) // 2
        y_off = (self.target_size - new_h) // 2
        canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized

        img_tensor = torch.from_numpy(canvas).unsqueeze(0).repeat(3, 1, 1)
        img_tensor = self.normalize(img_tensor)
        return img_tensor, canvas

    def __getitem__(self, index):
        path = self.filepaths[index]
        label = self.labels[index]
        is_dcm = path.suffix.upper() == ".DCM"

        if is_dcm:
            if not DICOM_AVAILABLE:
                raise RuntimeError("pydicom required for .dcm files. Run: !pip install pydicom")
            img_raw = dcmread(path).pixel_array
        else:
            img_raw = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img_raw is None:
                raise RuntimeError(f"Failed to read image: {path}")

        img_tensor, canvas = self.preprocess_image(img_raw, is_dcm=is_dcm)
        return img_tensor, label, str(path), canvas

In [ ]:
TARGET_SIZE = 448  # must be a multiple of DINOv2 patch size 14: 448 = 32 * 14
TRAIN_PATH = os.path.join(GRID_DIR, "train")
TEST_PATH  = os.path.join(GRID_DIR, "test")

train_dataset = IndustrialAnomalyDataset(TRAIN_PATH, target_size=TARGET_SIZE)
test_dataset  = IndustrialAnomalyDataset(TEST_PATH,  target_size=TARGET_SIZE)

print(f"Train samples: {len(train_dataset)}  (all 'good')")
print(f"Test samples:  {len(test_dataset)}")
print(f"Test label categories: {sorted(set(test_dataset.labels))}")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

# Shape verification
imgs, labels, paths, canvases = next(iter(train_loader))
print(f"\nBatch tensor shape: {tuple(imgs.shape)}  (expected: [8, 3, 448, 448])")
print(f"Label in batch: {set(labels)}  (should be: {{'good'}})")

## Step 3: DINOv2 Feature Extraction

DINOv2 ViT-S/14 divides each 448×448 image into non-overlapping 14×14 patches:
- **Grid**: 448 / 14 = 32 patches per side → **1024 patches per image**
- **Embedding dim**: 384 (ViT-Small)

We use `get_intermediate_layers(n=1)` to get the last transformer block's patch tokens.
Features are L2-normalized so cosine distance equals Euclidean distance on the unit sphere.

In [ ]:
import torch.nn.functional as F

print("Loading DINOv2 ViT-S/14 from facebookresearch/dinov2...")
print("(First run downloads ~100 MB weights; cached for the rest of this session)")
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
dinov2 = dinov2.to(device).eval()

PATCH_SIZE = dinov2.patch_size
GRID_DIM = TARGET_SIZE // PATCH_SIZE
N_PATCHES = GRID_DIM ** 2
EMBED_DIM = 384
print(f"\nPatch size: {PATCH_SIZE}px")
print(f"Patch grid: {GRID_DIM} x {GRID_DIM} = {N_PATCHES} patches per image")
print(f"Embedding dim: {EMBED_DIM}")

In [ ]:
all_features = []
total_batches = len(train_loader)
print(f"Extracting features from {len(train_dataset)} training images ({total_batches} batches)...")

with torch.no_grad():
    for i, (imgs, _, _, _) in enumerate(train_loader):
        imgs = imgs.to(device)
        # Shape: [batch, N_PATCHES, EMBED_DIM]
        features = dinov2.get_intermediate_layers(imgs, n=1)[0]
        # Flatten batch: [batch * N_PATCHES, EMBED_DIM]
        features = features.reshape(-1, EMBED_DIM)
        # L2 normalise so cosine distance = 1 - dot product
        features = F.normalize(features, p=2, dim=-1)
        all_features.append(features.cpu().numpy())
        if (i + 1) % 5 == 0 or (i + 1) == total_batches:
            print(f"  Batch {i+1}/{total_batches}")

normal_features = np.concatenate(all_features, axis=0)
print(f"\nFeature matrix shape: {normal_features.shape}")
print(f"  = {len(train_dataset)} images x {N_PATCHES} patches = {len(train_dataset)*N_PATCHES} total patches")

## Step 4: DP-GMM Clustering

Scikit-learn's `BayesianGaussianMixture` implements variational inference for the Dirichlet Process prior.

- **`n_components=150`** — upper bound; unused components are suppressed to near-zero weight
- **`weight_concentration_prior=1e-3`** — small α encourages sparsity (few active clusters)
- **`covariance_type="diag"`** — O(K·D) memory vs O(K·D²) for full; avoids singularity on high-dim data

After fitting, we keep only components with weight > 0.01 as the normal prototype library.

In [ ]:
from sklearn.mixture import BayesianGaussianMixture

print("Fitting variational DP-GMM on normal patch features...")
dpmm = BayesianGaussianMixture(
    n_components=150,
    covariance_type="diag",
    weight_concentration_prior=1e-3,
    max_iter=100,
    random_state=42,
)
dpmm.fit(normal_features)

active_mask = dpmm.weights_ > 0.01
centroids = dpmm.means_[active_mask]
print(f"\nConverged. Active components: {len(centroids)} / 150")
print(f"(Components below weight 0.01 are suppressed as 'empty')")
print(f"\nCentroid matrix shape: {centroids.shape}  [K_active, {EMBED_DIM}]")

## Step 5: Test-time Anomaly Scoring

For each test patch embedding:
1. Compute cosine similarity to all active centroids
2. Anomaly score = **1 − max cosine similarity** = cosine distance to nearest normal cluster
3. Reshape to 32×32 spatial grid, upsample to 448×448, apply Gaussian blur

High score = far from any normal pattern → anomalous region.

In [ ]:
import matplotlib.pyplot as plt

centroids_tensor = torch.from_numpy(centroids).to(device)
os.makedirs("figures", exist_ok=True)

TEST_LIMIT = 5
visualized = 0

with torch.no_grad():
    for imgs, labels, paths, canvases in test_loader:
        if visualized >= TEST_LIMIT:
            break

        label = labels[0]
        if label == "good":
            continue

        imgs = imgs.to(device)
        # Extract features for the single test image
        feats = dinov2.get_intermediate_layers(imgs, n=1)[0].squeeze(0)  # [N_PATCHES, EMBED_DIM]
        feats = F.normalize(feats, p=2, dim=-1)

        # Cosine distance to nearest centroid
        cos_sim = torch.matmul(feats, centroids_tensor.T)   # [N_PATCHES, K_active]
        min_dist, _ = torch.min(1.0 - cos_sim, dim=-1)      # [N_PATCHES]
        image_score = torch.max(min_dist).item()

        # Upsample patch grid to image resolution
        anomaly_map = min_dist.reshape(1, 1, GRID_DIM, GRID_DIM)
        anomaly_map = F.interpolate(
            anomaly_map, size=(TARGET_SIZE, TARGET_SIZE),
            mode="bicubic", align_corners=False
        ).squeeze().cpu().numpy()
        anomaly_map = cv2.GaussianBlur(anomaly_map, (15, 15), 0)

        # Visualize
        canvas = canvases[0].numpy()
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
        ax1.imshow(canvas, cmap="gray"); ax1.set_title("Input"); ax1.axis("off")
        ax2.imshow(anomaly_map, cmap="jet")
        ax2.set_title(f"Anomaly Map (score={image_score:.3f})"); ax2.axis("off")
        ax3.imshow(canvas, cmap="gray")
        ax3.imshow(anomaly_map, cmap="jet", alpha=0.5)
        ax3.set_title(f"Overlay | type: {label}"); ax3.axis("off")
        fig.suptitle(f"File: {os.path.basename(paths[0])}", fontsize=11)
        plt.tight_layout()
        plt.savefig(f"figures/defect_{visualized}.png", dpi=150)
        plt.show()

        visualized += 1

print(f"\nSaved {visualized} anomaly heatmaps to figures/")